# Imipramine CAS(6e,6o) - results and the presentation deck

The companion to [`colab.ipynb`](colab.ipynb). That notebook exists to **build**
the Hamiltonian; this one exists to **present** what came out of it.

It does three things the build notebook does not:

1. **Classical baselines.** MP2, CCSD and CCSD(T) frozen to the identical active
   space. Without them "ADAPT reached 0.18 mHa" has no scale - that number is
   excellent if CCSD(T) sits at 2 mHa and irrelevant if CCSD(T) sits at 0.02.
   It is the first thing a chemist asks and the deck should answer it.
2. **A gate-count sweep.** ADAPT appends, so every prefix of a finished run is a
   real ansatz. Transpiling nine of them turns one run into an accuracy-versus-
   cost curve.
3. **Five figures**, one per claim, sized for slides.

Everything here reads the cache. If you already have `hamiltonian_cas6e6o.json`
and its receipt, upload them in section 4 and skip the RHF entirely.

> **Read before presenting.** These are noiseless statevector results - no
> hardware, no shot noise. The published Helios-1 number was 200.4 mHa against
> the same paper's 1.15 mHa simulator result, so the hardware gap is two orders
> of magnitude and is not measured here. Say that yourself before someone asks.

## 1. Environment

Nothing below is Colab-specific - the same cells work in any Linux/macOS shell
with the `!` prefixes dropped.

In [ ]:
import platform, sys

print(f"Python   : {platform.python_version()}")
print(f"Platform : {platform.system()} {platform.machine()}")

major, minor = sys.version_info[:2]
if (major, minor) < (3, 10):
    print("\nWARNING: this workflow is developed on 3.12 and needs at least 3.10.")
elif (major, minor) >= (3, 14):
    print("\nWARNING: newer than the tested 3.12; wheels may be missing.")
else:
    print("\nVersion is fine.")

## 2. Install

`pyscf` is needed by two commands here - `prepare` and `classical` - and is the
one Windows cannot have. `matplotlib` is needed for the figures and is the only
new dependency relative to the build notebook.

**numpy and scipy are deliberately not listed.** Colab already ships versions
that satisfy everything here, and naming them explicitly makes pip resolve them
fresh to the newest release, which is how a working runtime gets broken.

In [ ]:
!pip install -q "pyscf>=2.5" "openfermion>=1.6,<2" "qiskit==2.5.*" "qiskit-aer==0.17.*" "matplotlib>=3.8"

In [ ]:
# Does the stack import and work? pip's resolver warnings do not answer that.
import importlib
from importlib import metadata

REQUIRED = ["numpy", "scipy", "pyscf", "openfermion", "qiskit", "qiskit_aer", "matplotlib"]
DISTRIBUTION = {"qiskit_aer": "qiskit-aer"}

missing = []
for module in REQUIRED:
    try:
        importlib.import_module(module)
        version = metadata.version(DISTRIBUTION.get(module, module))
        print(f"  ok        {module:<14} {version}")
    except Exception as error:
        missing.append(module)
        print(f"  MISSING   {module:<14} {type(error).__name__}")

print("\nAll present." if not missing else f"\nSTOP: {missing} did not import.")

## 3. Get the workflow

Set `REPO` to your fork if you are not running from the original.

In [ ]:
import os, shutil, subprocess
from pathlib import Path

REPO = "https://github.com/bhargav2603/fujitsu.git"
BRANCH = "main"
FOLDER = "imipramine_qiskit"

target = Path("/content/repo")
if target.exists():
    shutil.rmtree(target)          # always start from a clean clone
subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, str(target)],
    check=True,
)
os.chdir(target / FOLDER)
print("\nworking directory:", Path.cwd())
print("modules:", sorted(p.name for p in Path.cwd().glob("*.py")))

## 4. The Hamiltonian

Two routes. **Uploading is much faster** - the 45-atom RHF is the only slow step
in this notebook, and the cache carries its own specification hash, so copying
it between machines is safe by construction.

Run the upload cell if you have the two JSON files. Otherwise skip it and run
`prepare` below.

In [ ]:
# ROUTE A -- upload a cache you already built. Skip if you do not have one.
from pathlib import Path

try:
    from google.colab import files
    print("Select hamiltonian_cas6e6o.json and hamiltonian_cas6e6o.validated.json")
    for name, blob in files.upload().items():
        Path(name).write_bytes(blob)
        print(f"  wrote {name}  ({len(blob):,} bytes)")
except ImportError:
    print("Not on Colab -- copy the two JSON files next to run.py yourself.")

In [ ]:
# ROUTE B -- build it. Skip this and the next cell if you uploaded a cache.
# --diagnose runs RHF once and prints the frontier window without building.
!python run.py prepare --diagnose

In [ ]:
!python run.py prepare
!python run.py validate --number-penalty 0 1 4

## 5. Classical baselines - the comparison that decides the story

MP2, CCSD and CCSD(T), **frozen to the identical CAS(6e,6o) window**. Running
them on all 152 electrons would correlate 73 core orbitals CASCI never touched
and produce a number that is meaningless for comparison.

With six electrons in six orbitals FCI reaches hextuple excitations, so CCSD is
a genuine approximation here and CCSD(T) a genuine test. Which way it lands is
not predictable from the size alone - a pi-conjugated active space carries
static correlation that coupled cluster handles poorly.

**Do not pre-write your conclusion.** If CCSD(T) beats ADAPT, the honest framing
is *verification, not advantage* - still a true and defensible result, and far
better than a deck that quietly omitted the baseline. Read the number first.

This repeats the RHF, so it takes a few minutes. The correlated solves in six
orbitals are instant, and the timings are recorded for the cost chart.

In [ ]:
!python run.py classical

## 6. ADAPT-VQE

Two runs, because they answer different questions.

**`--max-operators 100`** is the one that converges. A 40-operator run stops on
the cap with the pool gradient still an order of magnitude above tolerance, and
an unconverged ADAPT state is not at the variational minimum - so it comes back
with a spin expectation around 6e-4 instead of ~1e-19, which blows the 0.16 mHa
contamination budget. The energy is fine; the run is still marked
`spin_contaminated`. Fitting the gradient decay (x0.95 per operator) puts the
tolerance crossing near operator 94, so 100 is the first cap worth trying.

**`--max-operators 31`** is the efficiency point. The 1.6 mHa target is met at
operator 28 and the published 1.15 mHa is beaten at 31, so every operator past
that buys accuracy nobody asked for at roughly 70 more two-qubit gates each.

`--pool uccsd` rather than the paper's `uccgsd`: at 12 qubits every operator
ADAPT selects from the 435-strong generalized pool already lives in the
63-operator uccsd subset, so the two give bit-identical energies while uccsd
builds twice as fast and emits 27% fewer two-qubit gates.

In [ ]:
!python run.py adapt --pool uccsd --max-operators 100

In [ ]:
!python run.py adapt --pool uccsd --max-operators 31

In [ ]:
# Optional contrast: the paper's pool, and the hardware-efficient ansatz.
# The HEA needs a number penalty -- use one that `validate` marked ok.
!python run.py adapt --pool uccgsd --max-operators 100
!python run.py vqe --method statevector --layers 4 --number-penalty 1

In [ ]:
!python run.py summary --sort error

## 7. The five figures

One figure per claim, in presentation order. Together they defend one sentence:

> ADAPT reproduces exact classical chemistry on a real tricyclic antidepressant
> at 12 qubits, for N times the published gate cost, while classical still wins
> outright at this size - and here is where that flips.

| # | Figure | The claim it defends |
|---|---|---|
| 1 | `error` | where the quantum number sits among classical methods |
| 2 | `gates` | what the circuit costs against the published 244 |
| 3 | `accuracy` | the two traded against each other - the chart that argues |
| 4 | `cost` | wall clock now, and where classical stops being free |
| 5 | `pipeline` | how the code runs, and the five places it can refuse |

`pareto_points` is the only slow parameter: one transpile per point, a second or
two each. Set it to 0 to skip the grown-ansatz curve - the chart still draws the
finished runs and the published point, which is most of the argument.

In [ ]:
%matplotlib inline
import importlib

import visualize
importlib.reload(visualize)   # so an edit to visualize.py lands without a restart

visualize.show_deck("results", pareto_points=9)

### The per-run diagnostic charts

The four convergence panels from the build notebook. These are backup slides:
useful when someone asks *how* it converged, wrong as a headline.

In [ ]:
visualize.show("results")

## 8. Download

Figures as 200 dpi PNGs plus every result JSON, in one zip.

In [ ]:
import shutil
from pathlib import Path

!python run.py deck --dpi 200
!python run.py plot

archive = Path("/content/imipramine_results")
if archive.exists():
    shutil.rmtree(archive)
shutil.copytree("results", archive)
for name in ("hamiltonian_cas6e6o.json", "hamiltonian_cas6e6o.validated.json"):
    source = Path(name)
    if source.exists():
        shutil.copy(source, archive / name)

bundle = shutil.make_archive("/content/imipramine_results", "zip", archive)
print(f"\n{bundle}  ({Path(bundle).stat().st_size:,} bytes)\n")
for path in sorted(archive.rglob("*")):
    if path.is_file():
        print(f"  {path.relative_to(archive)}")

try:
    from google.colab import files
    files.download(bundle)
except ImportError:
    pass